In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SILVER_TABLE = "workspace.silver.orders"

silver_df = spark.table(SILVER_TABLE)

print("Silver records:", silver_df.count())

In [0]:
display(silver_df.limit(10))

In [0]:
sales_df = (
    silver_df

    .withColumn(
        "gross_sales",
        F.col("quantity") * F.col("unit_price")
    )

    .withColumn(
        "discount_amount",
        F.col("gross_sales")
        * F.col("discount")
        / F.lit(100)
    )

    .withColumn(
        "net_sales",
        F.col("gross_sales")
        - F.col("discount_amount")
    )
)

In [0]:
display(
    sales_df.select(
        "order_id",
        "quantity",
        "unit_price",
        "discount",
        "gross_sales",
        "discount_amount",
        "net_sales"
    ).limit(20)
)

In [0]:
daily_sales_df = (
    sales_df
    .groupBy("order_date")
    .agg(
        F.countDistinct("order_id")
            .alias("total_orders"),

        F.sum("quantity")
            .alias("total_quantity"),

        F.sum("gross_sales")
            .alias("gross_sales"),

        F.sum("discount_amount")
            .alias("total_discount"),

        F.sum("net_sales")
            .alias("net_sales")
    )
    .orderBy("order_date")
)

In [0]:
display(daily_sales_df)

In [0]:
(
    daily_sales_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.gold.daily_sales"
    )
)

In [0]:
display(
    spark.table(
        "workspace.gold.daily_sales"
    )
)

In [0]:
category_sales_df = (
    sales_df
    .groupBy("category")
    .agg(
        F.countDistinct("order_id")
            .alias("total_orders"),

        F.sum("quantity")
            .alias("total_quantity"),

        F.sum("net_sales")
            .alias("net_sales")
    )
)

In [0]:
category_sales_df = (
    category_sales_df
    .withColumn(
        "average_order_value",
        F.when(
            F.col("total_orders") > 0,
            F.col("net_sales")
            / F.col("total_orders")
        ).otherwise(F.lit(0))
    )
)

In [0]:
display(category_sales_df)

In [0]:
(
    category_sales_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.gold.category_sales"
    )
)

In [0]:
display(
    spark.table(
        "workspace.gold.category_sales"
    )
)

In [0]:
country_sales_df = (
    sales_df
    .groupBy("country")
    .agg(
        F.countDistinct("order_id")
            .alias("total_orders"),

        F.sum("net_sales")
            .alias("net_sales")
    )
    .orderBy(
        F.col("net_sales").desc()
    )
)

In [0]:
display(country_sales_df)

In [0]:
(
    country_sales_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.gold.country_sales"
    )
)

In [0]:
category_rank_window = Window.orderBy(
    F.col("net_sales").desc()
)

In [0]:
category_rank_df = (
    category_sales_df
    .withColumn(
        "sales_rank",
        F.dense_rank().over(
            category_rank_window
        )
    )
    .orderBy("sales_rank")
)

In [0]:
display(category_rank_df)

In [0]:
display(
    spark.sql("""
        SELECT
            country,
            COUNT(DISTINCT order_id) AS total_orders,
            SUM(quantity) AS total_quantity,
            ROUND(SUM(net_sales), 2) AS net_sales
        FROM (
            SELECT
                *,
                quantity * unit_price AS gross_sales,
                quantity * unit_price
                    * discount / 100 AS discount_amount,
                quantity * unit_price
                    - quantity * unit_price
                    * discount / 100 AS net_sales
            FROM workspace.silver.orders
        )
        GROUP BY country
        ORDER BY net_sales DESC
    """)
)

In [0]:
display(
    spark.sql("""
        SELECT
            ROUND(
                SUM(quantity * unit_price),
                2
            ) AS silver_gross_sales,

            ROUND(
                SUM(
                    quantity
                    * unit_price
                    * discount / 100
                ),
                2
            ) AS silver_discount,

            ROUND(
                SUM(
                    quantity * unit_price
                    -
                    quantity
                    * unit_price
                    * discount / 100
                ),
                2
            ) AS silver_net_sales
        FROM workspace.silver.orders
    """)
)

In [0]:
display(
    spark.sql("""
        SELECT
            ROUND(SUM(gross_sales), 2)
                AS gold_gross_sales,

            ROUND(SUM(total_discount), 2)
                AS gold_discount,

            ROUND(SUM(net_sales), 2)
                AS gold_net_sales
        FROM workspace.gold.daily_sales
    """)
)

In [0]:
spark.sql("""
SHOW TABLES IN workspace.gold
""").show(truncate=False)

In [0]:
spark.sql("""
SELECT COUNT(*) FROM workspace.gold.daily_sales
""").show()

spark.sql("""
SELECT COUNT(*) FROM workspace.gold.category_sales
""").show()

spark.sql("""
SELECT COUNT(*) FROM workspace.gold.country_sales
""").show()